# 03 – Preprocessing and Feature Engineering

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from classification_utils import load_data, prepare_target, add_engineered_features, make_preprocessor

df = load_data()
df, feature_cols = prepare_target(df)

print("Missing values in predictors:")
display(df[feature_cols].isna().sum().sort_values(ascending=False).to_frame("missing_count").head(15))

print("Duplicate rows:", int(df.duplicated().sum()))

outlier_rows=[]
for col in feature_cols:
    s=df[col].dropna()
    q1,q3=s.quantile([.25,.75])
    iqr=q3-q1
    lo,hi=q1-1.5*iqr,q3+1.5*iqr
    out=((s<lo)|(s>hi)).sum()
    outlier_rows.append([col,q1,q3,lo,hi,out])
outliers=pd.DataFrame(outlier_rows,columns=["feature","Q1","Q3","lower","upper","outlier_count"]).sort_values("outlier_count",ascending=False)
display(outliers.head(15))

Missing values in predictors:


,missing_count
X_Minimum,97
Edges_Index,97
Luminosity_Index,97
Orientation_Index,97
Log_Y_Index,97
Log_X_Index,97
LogOfAreas,97
Outside_Global_Index,97
Edges_Y_Index,97
Edges_X_Index,97


Duplicate rows: 0


,feature,Q1,Q3,lower,upper,outlier_count
7,Sum_of_Luminosity,9580.7500,8.186000e+04,-9.883812e+04,1.902789e+05,377
4,Pixels_Areas,85.0000,8.602500e+02,-1.077875e+03,2.023125e+03,372
17,Outside_X_Index,0.0066,2.270000e-02,-1.755000e-02,4.685000e-02,350
5,X_Perimeter,15.0000,9.000000e+01,-9.750000e+01,2.025000e+02,320
13,Steel_Plate_Thickness,40.0000,8.000000e+01,-2.000000e+01,1.400000e+02,227
6,Y_Perimeter,13.0000,8.325000e+01,-9.237500e+01,1.886250e+02,170
9,Maximum_of_Luminosity,124.0000,1.400000e+02,1.000000e+02,1.640000e+02,138
25,Luminosity_Index,-0.1950,-6.642500e-02,-3.878625e-01,1.264375e-01,127
3,Y_Maximum,461824.7500,2.167777e+06,-2.097104e+06,4.726705e+06,81
2,Y_Minimum,468209.7500,2.182184e+06,-2.102751e+06,4.753144e+06,80


## Cleaning Strategy

Missing numeric values are median-imputed inside the model pipeline. Duplicate rows are checked and are not removed when none exist. IQR-based outlier clipping is used rather than deleting samples. All learned preprocessing parameters are fitted on the training set only.

## Feature Engineering

In [2]:
X = add_engineered_features(df[feature_cols])
print("Original features:", len(feature_cols))
print("Features after engineering:", X.shape[1])
display(X[["Plate_Width","Plate_Height","Perimeter_Ratio","Area_Perimeter_Ratio"]].head())

Original features: 27
Features after engineering: 31


,Plate_Width,Plate_Height,Perimeter_Ratio,Area_Perimeter_Ratio
0,8.0,44.0,0.386364,4.377049
1,6.0,29.0,0.333333,2.700000
2,NaN,18.0,0.421053,NaN
3,7.0,45.0,0.288889,3.034483
4,17.0,257.0,NaN,NaN


### Justification
`Plate_Width` and `Plate_Height` indicate the dimensions of defects. `Perimeter_Ratio` refers to the geometry related to boundaries, while `Area_Perimeter_Ratio` relates to the area and perimeter of defects. It could reveal useful shape information for classifiers.

## Stratified 80:20 Split

In [3]:
le = LabelEncoder()
y = le.fit_transform(df["Fault_Type"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Train proportions:")
display(pd.Series(le.inverse_transform(y_train)).value_counts(normalize=True).mul(100).round(2).to_frame("train_%"))
print("Test proportions:")
display(pd.Series(le.inverse_transform(y_test)).value_counts(normalize=True).mul(100).round(2).to_frame("test_%"))

preprocessor = make_preprocessor(X_train)
print("Numeric columns:", len(X_train.select_dtypes(include=np.number).columns))
print("Categorical columns:", len(X_train.select_dtypes(include=["object","category","bool"]).columns))

Train: (1552, 31)  Test: (389, 31)
Train proportions:


,train_%
Other_Faults,34.66
Bumps,20.68
K_Scatch,20.17
Z_Scratch,9.79
Pastry,8.12
Stains,3.74
Dirtiness,2.84


Test proportions:


,test_%
Other_Faults,34.70
Bumps,20.82
K_Scatch,20.05
Z_Scratch,9.77
Pastry,8.23
Stains,3.60
Dirtiness,2.83


Numeric columns: 31
Categorical columns: 0


### Observation

The 80:20 stratified split retains the approximate class representation in both data splits. The median imputation, the IQR truncation, and optionally the encoding and standardization are set up to be applied within each classifier pipeline, to avoid any test set leakage.